# Lab: elevation change from ICESat-2

```{note}
This lab accompanies {doc}`laser-altimetry`. It is **not executed at build time** — the cells with real data access require an Earthdata account and a local Python environment with `icepyx`, `earthaccess`, `h5py`, `numpy`, and `matplotlib` installed. Run it yourself, cell by cell, and read the explanatory sections before each code block.
```

A single ICESat-2 orbit deposits a ribbon of photon arrivals from pole to pole. By the time that ribbon becomes a number in a mass-balance budget it has passed through four conceptually distinct stages: raw photon timing, surface-height extraction, repeat-track differencing, and a density conversion. Each stage introduces assumptions that matter. The lab walks you through all four, using a real outlet glacier as the target, so that by the end you have both a working elevation-change estimate and a clear accounting of what that number actually means.

The arc is straightforward. Sections 1–2 cover where the data lives and how to retrieve a small spatial subset without drowning in gigabytes. Section 3 reads the raw photon cloud and plots it. Section 4 collapses that cloud into along-track surface heights by a windowed robust mean, a stripped-down version of the ATL06 algorithm. Section 5 differences two passes to get elevation change. Section 6 is the tasks. Section 7 asks what assumptions separate a photon cloud from a mass-balance statement.

```{admonition} Patterns from UW-GDA
:class: seealso
The data-access style of this lab, programmatic search and download, windowed reads, and working in sensible projections, follows the UW Geospatial Data Analysis course of {cite}`shean2023`, which teaches these patterns on ICESat laser altimetry over the western U.S. Module 3 (NumPy, Pandas, and Matplotlib) builds the array and time-series skills the photon-cloud reduction here relies on, and Module 4 (GeoPandas, CRS, and projections) covers the polar projection choices that matter when you put ICESat-2 points on a Greenland map. Work through those two modules there if the tooling here feels unfamiliar.
```

## 1. Data sources

ICESat-2 data are archived at the **National Snow and Ice Data Center Distributed Active Archive Center** (NSIDC DAAC), accessible through NASA's Earthdata Login system. Everything downstream — the Python clients, the subsetting services, the direct S3 access — authenticates against that one account. If you do not have one, register at [urs.earthdata.nasa.gov](https://urs.earthdata.nasa.gov) before continuing.

Two Python libraries make programmatic access manageable.

**`icepyx`** is purpose-built for ICESat-2: it handles the NSIDC Harmony subsetting API, constructs bounding-box and time queries, and returns spatially subsetted granules rather than forcing you to download full orbits {cite}`markus2017`. Version-sensitive lines below are marked `# verify: icepyx API` because the library's interface has changed across major versions and will likely change again.

**`earthaccess`** is a broader NASA data-access library that handles authentication, granule search, and direct S3 streaming for any Earthdata collection. It pairs well with `icepyx` for authentication.

### Data volumes and the discipline of subsetting

This is the part students tend to skip over, and they regret it. An ATL03 global granule — one roughly five-minute segment of one of ICESat-2's six beams — runs to **several gigabytes of HDF5**. A single day of data for all six beams is tens of gigabytes. A season over Greenland, unsubsetted, will fill a laptop's SSD before the science begins.

**Spatial and temporal subsetting is not optional.** The NSIDC Harmony service will clip granules to your bounding box server-side before transfer; use it. The workflow below requests only the granules that intersect a small polygon around the target glacier, and within those granules it requests only the variables needed.

A practical discipline for any ICESat-2 analysis:

- **Process granule-by-granule.** Open one file, extract the arrays you need, accumulate summary statistics or save a small derived product, then close the file and move on. Never read a season of ATL03 into memory at once.
- **Prefer ATL06 over ATL03 when surface heights suffice.** ATL06 (along-track land-ice height) has already done the photon-to-surface reduction. Each granule is tens of megabytes rather than gigabytes, and for most elevation-change work it is the right starting product {cite}`markus2017`.
- **Budget storage before you search.** Estimate granule count × mean granule size before issuing a bulk order. The NSIDC data portal shows granule sizes; check them.

This lab uses ATL03 for Section 3 (because the point is to see the photon cloud) and ATL06 for Section 5 (because repeat-track differencing on pre-reduced heights is computationally tractable and illustrates the real workflow).

## 2. Search and order a spatial subset with icepyx

The target glacier is **Helheim Glacier**, a large marine-terminating outlet on Greenland's southeast coast, chosen because it is fast-moving, well-studied, and consistently covered by ICESat-2 repeat tracks. The bounding box below clips to a roughly 50 km × 60 km window around the lower trunk and terminus.

The date range requests two repeat cycles separated by about a year — enough time to accumulate a measurable elevation signal over the ablation zone without worrying about the seasonal cycle cancelling itself out.

In [ ]:
import icepyx as ipx  # verify: icepyx API

# Helheim Glacier lower trunk bounding box [lon_min, lat_min, lon_max, lat_max]
spatial_extent = [-38.8, 66.2, -38.0, 66.7]  # verify: icepyx API

# Two ICESat-2 repeat cycles about one year apart
date_range = ['2020-05-01', '2020-05-31']  # verify: icepyx API

# ATL03 global geolocated photon heights
region_atl03 = ipx.Query(  # verify: icepyx API
    dataset='ATL03',
    spatial_extent=spatial_extent,
    date_range=date_range,
)

print('ATL03 granules found:', region_atl03.avail_granules(ids=True))  # verify: icepyx API

# ATL06 land-ice heights for the repeat-track differencing in Section 5
date_range_prior = ['2019-05-01', '2019-05-31']  # verify: icepyx API
region_atl06_2019 = ipx.Query('ATL06', spatial_extent, date_range_prior)  # verify: icepyx API
region_atl06_2020 = ipx.Query('ATL06', spatial_extent, date_range)  # verify: icepyx API

print('ATL06 granules (2019):', region_atl06_2019.avail_granules(ids=True))  # verify: icepyx API
print('ATL06 granules (2020):', region_atl06_2020.avail_granules(ids=True))  # verify: icepyx API

In [ ]:
import earthaccess

# Authenticate via .netrc file (recommended) or interactive prompt.
# earthaccess writes credentials to ~/.netrc on first login so you
# do not have to type them again.
earthaccess.login(strategy='netrc')  # verify: icepyx API

# Request spatially subsetted download — the NSIDC service clips
# each granule to the bounding box before transfer.
# Adjust the output path to a directory with at least 2 GB free.
OUTPUT_DIR = './data/helheim/'  # edit this path for your machine

region_atl03.order_granules(verbose=True)  # verify: icepyx API
region_atl03.download_granules(OUTPUT_DIR)  # verify: icepyx API

region_atl06_2019.order_granules(verbose=True)  # verify: icepyx API
region_atl06_2019.download_granules(OUTPUT_DIR)  # verify: icepyx API

region_atl06_2020.order_granules(verbose=True)  # verify: icepyx API
region_atl06_2020.download_granules(OUTPUT_DIR)  # verify: icepyx API

## 3. Read ATL03 photon heights and plot the cloud

ATL03 stores its data in HDF5 groups, one per beam. Each beam group contains `heights/h_ph` (photon height above the WGS84 ellipsoid, in metres), `heights/dist_ph_along` (along-track distance since the segment start), and `heights/signal_conf_ph` (a per-photon signal confidence flag). The confidence flags run from $-2$ (noise) to 4 (high-confidence signal), and the atmosphere flags identify specular cloud returns that sit above the surface.

When you plot an unfiltered photon cloud over a glacier you see three populations stacked vertically. Solar background noise scatters uniformly through roughly 60 m of altitude around the surface return window. Blowing-snow and thin-cloud returns cluster above the surface. True surface returns trace a thin, dense ribbon, curved along the glacier's surface slope. The job of ATL06 is to find that ribbon.

In [ ]:
import glob
import h5py
import numpy as np
import matplotlib.pyplot as plt

# Pick the first ATL03 granule that was downloaded.
atl03_files = sorted(glob.glob(OUTPUT_DIR + 'ATL03_*.h5'))
print('ATL03 files found:', atl03_files)
atl03_path = atl03_files[0]  # edit index if you want a different granule

# ICESat-2 beam naming: gt1l, gt1r, gt2l, gt2r, gt3l, gt3r
# Strong beams (the ones to prefer for surface-height retrieval)
# are labelled in the ancillary data; we just pick gt1l here.
BEAM = 'gt1l'

with h5py.File(atl03_path, 'r') as f:
    h_ph   = f[f'{BEAM}/heights/h_ph'][:]          # photon heights, m (ellipsoidal)
    x_ph   = f[f'{BEAM}/heights/dist_ph_along'][:]  # along-track distance, m
    conf   = f[f'{BEAM}/heights/signal_conf_ph'][:, 0]  # land-ice confidence column

# Split by signal confidence.
# conf >= 2 is the threshold commonly used for surface photons.
is_signal = conf >= 2
is_noise  = conf < 2

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(x_ph[is_noise]  / 1e3, h_ph[is_noise],  s=0.5, c='#bbb', alpha=0.3, label='noise')
ax.scatter(x_ph[is_signal] / 1e3, h_ph[is_signal], s=0.8, c='#1f77b4', alpha=0.6, label='signal')
ax.set_xlabel('along-track distance (km)')
ax.set_ylabel('height above ellipsoid (m)')
ax.set_title(f'ATL03 photon cloud — {BEAM} — Helheim Glacier')
ax.legend(markerscale=4)
plt.tight_layout()
plt.show()

print(f'Total photons:   {len(h_ph):>9,}')
print(f'Signal photons:  {is_signal.sum():>9,}  ({100*is_signal.mean():.1f}%)')
print(f'Height range:    {h_ph.min():.1f} – {h_ph.max():.1f} m')

## 4. Reduce photons to along-track surface heights

The ATL06 algorithm divides each beam into 40 m along-track segments and fits a linear height model to the signal photons within each segment, iteratively rejecting outliers. The formal version involves least-squares fitting, sensitivity-dependent weights, and several quality flags. The version here is simpler but captures the essential idea: slide a window along the track, take the median of signal-photon heights within it (the median is robust against the remaining noise and blowing-snow photons), and assign the result to the window centre.

The window length is a tunable parameter and Task 1 asks you to explore its effect. Too short and you have too few photons per window for a stable estimate; too long and you average over real surface curvature.

In [ ]:
def photon_to_surface(x_ph, h_ph, conf, window_m=100.0, conf_thresh=2, min_photons=5):
    """Windowed robust-median reduction of a photon cloud to surface heights.

    Parameters
    ----------
    x_ph        : along-track distance, m (sorted ascending)
    h_ph        : photon height above ellipsoid, m
    conf        : per-photon signal confidence flag (ATL03 land-ice column)
    window_m    : along-track window length, m
    conf_thresh : minimum confidence flag to include a photon
    min_photons : minimum photons required to produce an estimate

    Returns
    -------
    x_seg : window-centre along-track positions, m
    h_seg : median surface height within each window, m
    n_seg : photon count per window
    """
    mask = conf >= conf_thresh
    xg, hg = x_ph[mask], h_ph[mask]

    x_start = xg[0]
    x_end   = xg[-1]
    centres = np.arange(x_start + window_m / 2.0, x_end, window_m)

    x_seg = []
    h_seg = []
    n_seg = []
    half = window_m / 2.0
    for xc in centres:
        idx = (xg >= xc - half) & (xg < xc + half)
        n = idx.sum()
        if n >= min_photons:
            x_seg.append(xc)
            h_seg.append(np.median(hg[idx]))
            n_seg.append(n)

    return np.array(x_seg), np.array(h_seg), np.array(n_seg)


x_seg, h_seg, n_seg = photon_to_surface(x_ph, h_ph, conf, window_m=100.0)

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axes[0].scatter(x_ph[is_signal] / 1e3, h_ph[is_signal], s=0.5, c='#aac',
                alpha=0.4, label='signal photons')
axes[0].plot(x_seg / 1e3, h_seg, 'r-', lw=1.2, label='windowed median (100 m)')
axes[0].set_ylabel('height (m)')
axes[0].legend(markerscale=4)

axes[1].plot(x_seg / 1e3, n_seg, 'k-', lw=0.8)
axes[1].set_xlabel('along-track distance (km)')
axes[1].set_ylabel('photons per window')

plt.suptitle('ATL03 photon cloud reduced to surface heights')
plt.tight_layout()
plt.show()

print(f'Surface segments: {len(x_seg)}')
print(f'Mean photons/window: {n_seg.mean():.1f}')

## 5. Repeat-track differencing and the firn/density caveat

Two passes over the same track separated by time $\Delta t$ give an elevation-change rate $\dot h = \Delta h / \Delta t$ at each along-track position. In practice, ICESat-2's ground tracks repeat to within a few metres, so a nearest-neighbour or linear interpolation between the two height profiles gives a usable difference without full co-registration. For this section we use the pre-reduced ATL06 heights because they are already on uniform 20 m segments and their associated quality flags make outlier rejection straightforward.

**The firn and density caveat** is the most important qualification on any elevation-change rate from laser altimetry. A surface height change $\Delta h$ is not a mass change until you know the density of the material that appeared or disappeared. Over the accumulation zone, inter-annual variability in snow accumulation and in firn compaction rates can shift the surface by tens of centimetres with no net mass change whatsoever. Over the ablation zone the surface is ice, and the density conversion is straightforward (roughly 917 kg m$^{-3}$), but even there a brief summer snowfall can offset the laser measurement from the ice surface {cite}`herron1980`.

Converting $\dot h$ to a mass-balance rate requires a model of the firn column. The standard approach is a firn densification model driven by reanalysis temperature and accumulation, which provides a firn-air-content correction that partitions the observed $\dot h$ into a firn-compaction component and a true mass-change component. That correction is applied to the ATL06-derived elevation-change maps before any mass budget is closed — see the discussion in {doc}`../foundations/snow-to-ice` for the physics, and {doc}`../ice_flow/mass-balance` for how it enters the balance.

In [ ]:
def read_atl06_track(filepath, beam='gt1l'):
    """Read ATL06 along-track land-ice heights for one beam.

    Returns along-track distance (m), height (m), and quality flag arrays.
    """
    with h5py.File(filepath, 'r') as f:
        grp = f[f'{beam}/land_ice_segments/']
        x = grp['ground_track/x_atc'][:]        # along-track coordinate, m
        h = grp['h_li'][:]                       # land-ice height, m
        q = grp['atl06_quality_summary'][:]      # 0 = good, 1 = use with caution
    return x, h, q


atl06_2019_files = sorted(glob.glob(OUTPUT_DIR + 'ATL06_2019*.h5'))
atl06_2020_files = sorted(glob.glob(OUTPUT_DIR + 'ATL06_2020*.h5'))

# If the Harmony subsetting returned multiple granules, concatenate them.
# Process one file at a time (not slurped into a season array).
def load_atl06_concat(file_list, beam='gt1l'):
    xs, hs = [], []
    for fp in file_list:
        x, h, q = read_atl06_track(fp, beam=beam)
        good = (q == 0) & np.isfinite(h)
        xs.append(x[good])
        hs.append(h[good])
    return np.concatenate(xs), np.concatenate(hs)


x19, h19 = load_atl06_concat(atl06_2019_files)
x20, h20 = load_atl06_concat(atl06_2020_files)

# Sort by along-track position before interpolation.
i19 = np.argsort(x19);  x19, h19 = x19[i19], h19[i19]
i20 = np.argsort(x20);  x20, h20 = x20[i20], h20[i20]

# Interpolate 2019 heights onto 2020 x-positions.
# Only where 2020 positions fall within the 2019 track extent.
in_range = (x20 >= x19[0]) & (x20 <= x19[-1])
h19_interp = np.interp(x20[in_range], x19, h19)
dh = h20[in_range] - h19_interp
x_diff = x20[in_range]

# 1-year separation assumed; adjust dt_yr if your date ranges differ.
dt_yr = 1.0
dh_rate = dh / dt_yr   # m/yr, positive = surface raised

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axes[0].plot(x19 / 1e3, h19, lw=0.8, label='May 2019')
axes[0].plot(x20 / 1e3, h20, lw=0.8, label='May 2020')
axes[0].set_ylabel('height (m)')
axes[0].legend()

axes[1].axhline(0, color='k', lw=0.5)
axes[1].plot(x_diff / 1e3, dh_rate, lw=0.8, color='#d62728')
axes[1].set_xlabel('along-track distance (km)')
axes[1].set_ylabel('dh/dt (m/yr)')
axes[1].set_title('Elevation change rate — Helheim Glacier (ATL06 repeat track)')

plt.tight_layout()
plt.show()

print(f'Along-track range: {x_diff[0]/1e3:.1f} – {x_diff[-1]/1e3:.1f} km')
print(f'Mean dh/dt: {np.nanmean(dh_rate):.2f} m/yr')
print(f'Std  dh/dt: {np.nanstd(dh_rate):.2f} m/yr')

In [ ]:
# Sketch of the density conversion.
# Over the terminus ablation zone we assume bare ice density.
# Over the accumulation zone the firn correction matters enormously;
# here we just flag the range of plausible corrections.

rho_ice   = 917.0   # kg/m^3
rho_water = 1000.0  # kg/m^3

# Bare-ice assumption: mass change rate (kg/m^2/yr = mm w.e./yr * rho_w)
dM_ice = rho_ice * np.nanmean(dh_rate)   # kg/m^2/yr

# Firn column could account for 0–0.6 m/yr of the observed dh;
# the sign and magnitude come from a firn model (herron1980 physics).
# We bracket the uncertainty rather than applying a specific model.
dh_firn_range = np.array([0.0, 0.6])   # m/yr, firn compaction contribution
dM_range = rho_ice * (np.nanmean(dh_rate) - dh_firn_range)

print('Density conversion sketch (ablation-zone assumption)')
print(f'  dh/dt (mean): {np.nanmean(dh_rate):.2f} m/yr')
print(f'  dM/dt (bare ice): {dM_ice:.0f} kg/m^2/yr')
print(f'  dM/dt range with firn correction: {dM_range[1]:.0f} to {dM_range[0]:.0f} kg/m^2/yr')
print()
print('See {doc}`../foundations/snow-to-ice` for firn densification physics.')
print('See {doc}`../ice_flow/mass-balance` for how dM enters the mass budget.')

## 6. Tasks

**Task 1 — window length sensitivity.** Run `photon_to_surface` with window lengths of 40 m, 100 m, 250 m, and 500 m on the same ATL03 granule. For each, plot the resulting height profile and compute the standard deviation of the segment heights. Explain, in one paragraph, the trade-off you see between noise level and spatial resolution, and identify which window length most closely reproduces the ATL06 20 m product in terms of along-track sampling.

**Task 2 — median versus mean.** Replace `np.median` in `photon_to_surface` with `np.mean` and repeat the 100 m reduction. Compute the root-mean-square difference between the two estimates at each window centre. Where on the track is the difference largest, and what does that tell you about the noise-photon population at those locations?

**Task 3 — crossover geometry.** ICESat-2's three beam pairs produce six tracks per pass; ascending and descending orbits cross each other at angles that depend on latitude. At a crossover point two tracks from different dates sample the same surface location, giving an elevation-change estimate free of the along-track interpolation uncertainty. Write down the geometry: if two tracks cross at an angle $\alpha$ and each has along-track noise $\sigma_h$, what is the uncertainty in the crossover elevation-difference estimate as a function of $\alpha$? At what crossing angle does the crossover method become less precise than the repeat-track interpolation used in Section 5, if the repeat-track interpolation distance is 10 m? (This task is analytical — no new code needed.)

**Task 4 — seasonal window choice.** The two ATL06 passes used in Section 5 are both from May. Explain why matching the seasonal phase of the two passes matters for an elevation-change measurement, using the firn densification argument of Section 5. What would go wrong if one pass were from May and the other from September?

## 7. Synthesis

You started with a file of photon arrival times and ended with a number in metres per year. Between those two points, five assumptions quietly accumulated, and it is worth laying them out explicitly because each one is a place where the measurement can mislead you.

The first assumption is that the signal-confidence flags correctly partition signal from noise. They do well in the ablation zone, where the dark ice surface is a strong reflector and the photon cloud is dense. They do worse over bright firn, where solar background is high and the surface return is smeared by forward scattering in the snow grains. Any segment count below about ten photons should be treated with suspicion regardless of the flag.

The second assumption is that the windowed median (or the ATL06 linear fit) captures the true surface height and not a bias introduced by off-nadir scattering, surface roughness at the footprint scale, or the asymmetric photon distribution near crevasse edges. The ATL06 algorithm applies a slope correction that the simple median does not, and over steeply sloping glacier tongues that correction is not negligible {cite}`markus2017`.

The third assumption is that two tracks labelled as repeat tracks actually sample the same surface. ICESat-2's pointing control is excellent but not perfect; the actual cross-track offset between two nominally repeat passes can be a few metres, and over a sloping surface a few metres of cross-track offset translates directly into a spurious height difference. The repeat-track processor accounts for this with a slope correction derived from the across-track beam pair. The simple interpolation in Section 5 does not, which is one reason its scatter is larger than the official ATL06 difference product.

The fourth assumption is temporal: the two passes represent the same point in the seasonal cycle. A May-to-May difference largely cancels the seasonal firn signal. A May-to-September difference measures mostly seasonal accumulation and melt, not secular change.

The fifth and deepest assumption is the density conversion. An elevation change of one metre has very different mass implications depending on whether it represents compacted firn, fresh snow, or melted ice. Over the ablation zone the ice-density assumption is reasonable for annual means. Over the accumulation zone, the firn correction from a model like {cite}`herron1980` can reverse the sign of the apparent mass trend — a glacier that is thinning geometrically can be gaining mass if the thinning is driven by firn compaction in response to a warm summer. This is not a pathological edge case; it happens regularly over high-elevation Antarctic and Greenland interior regions.

What separates the photon cloud from a mass-balance statement is these five assumptions, plus the integration over area (a single track is not a glacier), plus the time-averaging needed to separate secular trends from interannual noise. The laser altimeter is the sharpest elevation tool we have from space, but getting from elevation to mass requires every one of these steps to be done carefully — which is why the discussions in {doc}`../foundations/snow-to-ice` and {doc}`../ice_flow/mass-balance` matter as much as the signal-processing of this lab.